# Method discovery — crop1.5

Exactly **C_SCJ**, **P3_MC**, **P3_SWS**; one predefined configuration per hypothesis. Notebook13/15 infrastructure is reused. Previous notebooks/resources remain unchanged.

1. Edit Kaggle resource/image/model paths and `RUN_FILTER`: `None`, `['C_SCJ']`, `['P3_MC']`, `['P3_SWS']`, or any valid subset. Runs have independent outputs/resume.
2. Execute setup and lightweight sanity checks. Review pending report/export before training.
3. The final runner starts expensive GPU training. Each completed run gets its standalone ZIP immediately; overall artifacts refresh after each completion.

SCJ trains with random context scale but **all Val3K/LCC evaluation uses crop1.5**. MC preserves hard worst selection; SWS samples exactly one detached spectral view/sample using a separate RNG. P3 epoch1 is clean-only. **Combined AUC alone** determines the frozen gate at **0.844642**. No CelebA full Test, detector, target fitting, extra search or automatic follow-up.


In [ ]:
# Editable Kaggle paths; reuse the frozen micro/pooled-LCC resources.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/16_micro_method_discovery_crop15'
LOG_EVERY_BATCHES=10
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=1.5
TRAIN_CROP_FACTOR=1.5
EVAL_CROP_FACTOR=1.5
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12



RUN_FILTER=None  # None or any non-empty unique subset of C_SCJ, P3_MC, P3_SWS.
MC_BETA=.10
SWS_TAU=1.0
SCJ_SCALE_RANGE=(1.30,1.70)
SCJ_SCALE_SEED=TRAIN_SEED+700000
SWS_SELECTOR_SEED=TRAIN_SEED+23000
REFERENCE_CROP=2.7
SCALED_CENTERS=[c*(1.5/2.7) for c in (.15,.45,.75)]
SCALED_SIGMA=.10*(1.5/2.7)
RUNS={
 'C_SCJ':{'family':'C','method':'CLEAN','crop_factor':1.5,'geometry_mode':'STATIC','centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'clean_ce_weight':1.,'spectral_ce_weight':0.,'scj_scale_range':list(SCJ_SCALE_RANGE)},
 'P3_MC':{'family':'P3','method':'MC','crop_factor':1.5,'geometry_mode':'STATIC','centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25,'mc_beta':MC_BETA},
 'P3_SWS':{'family':'P3','method':'SWS','crop_factor':1.5,'geometry_mode':'STATIC','centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'clean_ce_weight':.75,'spectral_ce_weight':.25,'sws_tau':SWS_TAU},
}
REPORT_STEM='method_discovery'
def validate_requested_filter(run_filter):
    if run_filter is not None and (not isinstance(run_filter,(list,tuple)) or not run_filter or len(set(run_filter))!=len(run_filter) or set(run_filter)-set(RUNS)):
        raise ValueError('RUN_FILTER must be None or a non-empty unique subset of C_SCJ, P3_MC, P3_SWS')
def select_runs(run_filter):
    validate_requested_filter(run_filter)
    return [name for name in RUNS if run_filter is None or name in run_filter]


In [ ]:
PROTOCOL_TEXT='# Method discovery crop1.5 protocol v1\n\nExactly three predefined hypotheses: C_SCJ tests occupancy/context sensitivity through training-only Uniform(1.30,1.70) crop scale; P3_MC tests signed-margin preservation with beta.10 and detached clean target; P3_SWS tests hard-selector brittleness with detached softmax(-margin/1.0) categorical one-view selection. No parameter search, DSF, R7, extra losses or runs.\n\nPrimary implementation references: notebooks13/15; notebook06 records historical P3-family methods, notebook12 records crop controls. All remain unchanged. Existing crop helper, pretrained MiniFASNetV2/128D/PAD/aux heads, manifests, augmentation, bbox jitter, metric/coverage/pooled prediction and checkpoint infrastructure are reused.\n\nFrozen Train10K/Val3K, DATA_SEED43/TRAIN_SEED100, BGR float32[0,1]3x80x80, batch256 AMP, SGD(lr.005,momentum.9,weight_decay5e-4), max20, milestones[6,14], gamma.2, early-stop earliest8/patience4. Min Val3K ACER with higher AUC tie-break. C_SCJ clean CE+aux from epoch1. P3 methods epoch1 clean CE+aux only, excluded from best/patience; later .75 clean+.25 selected spectral CE+aux, plus .10 MC only in P3_MC. No new stopping policy.\n\nAux unchanged: .1 spoof CE+.1 lighting CE+1.0 Real-only attributes BCE, clean only. P3 static SF centers [.15,.45,.75]*(1.5/2.7), sigma.10*(1.5/2.7), Uniform gain(.65,.90), shared across bands/channels, existing FFT/DC/clipping. MC calls original hard worst helper unchanged; SWS eval/no_grad selection constructs one detached view/sample, then only clean+one spectral gradient forward. No gradients through selector. Clean target detached in MC; selected spectral margin gradient remains enabled.\n\nSCJ scale drawn after exact bbox jitter via separate Random(SCJ_SCALE_SEED+epoch*100000+index), SCJ_SCALE_SEED=TRAIN_SEED+700000. This indexed stream has no evolving state; checkpoint records/validates seed/policy and resumes epoch/index exactly. Augmentation/bbox/sample order/gain streams remain independent. SWS uses separate torch.Generator with SWS_SELECTOR_SEED=TRAIN_SEED+23000; its state is saved/restored with model/optimizer/scheduler/scaler/best/patience/history/global and gain RNG states. Runs isolated by exact run IDs. No historical constants enter training/config/RNG.\n\nAll Val3K/LCC eval always crop1.5. Reload best; recompute/freeze Val3K minACER threshold; infer physical LCC training/development/evaluation, concatenate predictions and recompute Combined metrics over valid scores with failed-candidate coverage intact. No target selection/stopping/calibration/loss adaptation, no CelebA full Test or detector. LCC manifests/caches are loaded as established resource metadata; image inference is only post-training after source freeze.\n\nVerified historical reporting values supplied by user: C_crop15 Val ACER/AUC .021034/.996635, LCC eval AUC/HTER .775915/.294045, combined AUC/EER/HTER/TPR1% .830642/.244741/.245404/.196291; P3_SF_crop15 .022508/.996631, .782407/.298970, .837525/.239614/.252629/.202231.\n\nFixed engineering gate ONLY Combined AUC: >=.844642 STRONG_PASS; >.837525 and <.844642 NEAR_MISS; otherwise NO_STRONG_PASS. Deltas versus C .830642 and SF .837525 reported absolute and percentage points. Secondary endpoints and all diagnostics are descriptive only. Highest passing Combined AUC ranks the next candidate; ties at six decimal reporting precision require manual review. No significance/final winner claim. No automatic follow-up: C_SCJ-only strongest suggests one future P3_SF_SCJ; passing MC/SWS suggests itself for100K; both SCJ and P3 passing permit at most one SCJ+best passing P3 combination, recommendation only. No pass means manual review, no threshold chasing.\n\nPer-epoch exact sample-pooled diagnostics: SCJ requested/realized mean/std/min/p05/median/p95/max, boundary/translation/reduction and center offset; MC mean loss, positive-only gap mean/median/max, active fraction, gradient-forward clean/worst means; SWS probabilities, entropy mean/median, sampled/hard fractions/agreement. Standard CE/band/margin/drop/flip/gain/crop diagnostics retained. These are not causal proof.\n\nEach completed run immediately gets artifacts and /kaggle/working/<run_id>.zip. Overall method_discovery_comparison.csv, method_discovery_gate.json, method_discovery_report.md and /kaggle/working/16_micro_method_discovery_crop15.zip are refreshed after each run. RUN_FILTER=None or any nonempty unique subset; RESUME=True. Editable Kaggle paths. No shared resume state. Source-only sanity/export cell before expensive final runner.\n\nValidation: parse/compile/cleared outputs; deterministic scale replay/independence/eval15; MC beta0 scalar/backward, detached teacher and enabled student, warmup; BN-safe selectors and exactly one spectral gradient batch; categorical probability/selector state replay; no historical/target fitting; Combined-only gate boundaries; isolated completed-run skip and ZIP. Small CPU fixtures only; full GPU execution/resource paths/image availability/T4 memory require Kaggle verification.\n\nLimitations: micro Train10K/Val3K, one seed, repeatedly used LCC external development benchmark; no statistical significance claim; engineering gate only; 100K/full-scale validation not performed; multi-seed robustness future work due to compute limits.\n'

import hashlib, time, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display
from PIL import Image

AUG_SEED=TRAIN_SEED+300000
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
validate_requested_filter(RUN_FILTER)
print('[Setup] Selected runs:',select_runs(RUN_FILTER),flush=True)
if (TRAIN_CROP_FACTOR,EVAL_CROP_FACTOR)!=(1.5,1.5) or any(spec['crop_factor']!=1.5 for spec in RUNS.values()):raise ValueError('This round uses crop 1.5 only')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen micro recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',
             root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()

def geometry_summary(g):
    return {'number_of_evaluated_samples':len(g),'boundary_clipped_fraction':float(g[:,0].mean()),
      'mean_crop_width_over_image_width':float(g[:,1].mean()),'mean_crop_height_over_image_height':float(g[:,2].mean()),
      'median_crop_width_over_image_width':float(np.median(g[:,1])),'median_crop_height_over_image_height':float(np.median(g[:,2])),
      'effective_scale_mean':float(g[:,3].mean()),'effective_scale_std':float(g[:,3].std()),
      'window_translation_fraction':float(g[:,4].mean()),'actual_scale_reduction_fraction':float(g[:,5].mean())}

def geometry_vector(g):
    return np.asarray([g['boundary_interaction'],g['width_ratio'],g['height_ratio'],g['effective_scale'],g['window_translated'],g['scale_reduced']],dtype=np.float64)

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE,return_geometry=False,geometry_only=False):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    initial_window=(left,top,right,bottom)
    requested_w,requested_h=box_w*float(scale),box_h*float(scale)
    requested_boundary=(cx-requested_w/2<0 or cy-requested_h/2<0 or cx+requested_w/2>src_w-1 or cy+requested_h/2>src_h-1)
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    geometry={'effective_scale':float(used),'window_translated':bool(any(abs(a-b)>1e-12 for a,b in zip(initial_window,(left,top,right,bottom)))),
      'scale_reduced':bool(used<float(scale)-1e-12),'boundary_interaction':bool(requested_boundary or used<float(scale)),
      'width_ratio':(rx-lx+1)/src_w,'height_ratio':(by-ty+1)/src_h}
    if geometry_only:return geometry
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    resized=cv2.resize(patch,(size,size))
    return (resized,geometry) if return_geometry else resized

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(centers,sigma,size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(centers,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*sigma**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def bbox_jitter(rec,rng):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if rng.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=rng.uniform(.95,1.05)
        cx+=rng.uniform(-.05,.05)*base;cy+=rng.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]

def copied_aug_bgr_seeded(patch,seed):
    # Augmentation RNG cannot consume bbox or spectral RNG state.
    random.seed(seed);np.random.seed(seed)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(seed)
    return copied_aug_bgr(patch)

def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('mini_train10k');val=load_frame('mini_val3k')
if (len(train),len(val))!=(10000,3000):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if 'sampling_seed' in frame and set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen micro seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen mini split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen micro batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'METHOD_DISCOVERY_CROP15_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model():
    seed_all(TRAIN_SEED)
    model=PADNet(True)
    raw=torch.load(pretrained,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Official PAD checkpoint format differs')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):
        raise RuntimeError('Official PAD checkpoint architecture differs')
    compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
    target=model.trunk.state_dict()
    if set(compatible)!=set(target) or any(tuple(compatible[k].shape)!=tuple(target[k].shape) for k in compatible):
        raise RuntimeError('PAD pretrained trunk keys differ')
    model.trunk.load_state_dict(compatible,strict=True)
    return model.to(device)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,epoch=0,crop_factor=1.5,scj=False):
        self.rows=frame.reset_index(drop=True);self.training=training;self.epoch=epoch;self.crop_factor=crop_factor;self.scj=scj
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        box=bbox_jitter(rec,random.Random(BBOX_SEED+self.epoch*100000+i)) if self.training else rec['bbox_xyxy']
        requested=training_crop_scale(self.training,self.epoch,i,self.scj)
        patch,metadata=mini_crop_bgr(image,box,scale=requested,return_geometry=True)
        geometry=np.r_[geometry_vector(metadata),requested]
        if self.training:patch=copied_aug_bgr_seeded(patch,AUG_SEED+self.epoch*100000+i)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key,geometry

class LCCData(Dataset):
    def __init__(self,split,crop_factor):
        self.split=split;self.crop_factor=crop_factor;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        patch,metadata=mini_crop_bgr(image,rec['bbox_xyxy'],scale=self.crop_factor,return_geometry=True)
        return tensor_of(patch),int(row.label),key,geometry_vector(metadata)

def loader(ds,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      drop_last=False,generator=gen)

def infer(model,dl,lcc=False,label='Evaluation'):
    model.eval();keys=[];ys=[];zs=[];geometry=[];started=time.monotonic()
    print('['+label+'] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc:x,y,k,g=batch
            else:x,y,_,_,_,_,k,g=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy());geometry.append(g.numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):
                print(f'[{label}] batch {step}/{len(dl)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs),np.concatenate(geometry)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))





def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'mini_train10k.csv','val_manifest':'mini_val3k.csv',
      'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'train_crop_factor':spec.get('scj_scale_range',spec['crop_factor']),'eval_crop_factor':EVAL_CROP_FACTOR,
      'base_order':'torch Generator(TRAIN_SEED+epoch)','bbox_seed':BBOX_SEED,'augmentation_seed':AUG_SEED,'gain_seed':GAIN_SEED,
      'scj_rng':{'seed':SCJ_SCALE_SEED,'policy':'indexed seed+epoch*100000+sample_index'},'sws_selector_seed':SWS_SELECTOR_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,'num_workers':NUM_WORKERS,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':0 if spec['method']=='CLEAN' else WARMUP_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':spec['centers'],'sigma':spec['sigma'],'gain':[.65,.90],'selection_chunk_size':SELECTION_CHUNK_SIZE,'preserve_dc':CSMR_PRESERVE_DC,'geometry_mode':spec['geometry_mode'],'reference_crop':REFERENCE_CROP},
      'input':'80x80 BGR float [0,1]','augmentation_api':AUGMENTATION_API,'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked,crop_factor,crop_diagnostics):
    scored_predictions=[];all_geometry=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z,g=infer(model,loader(LCCData(split,crop_factor)),True,label=folder.name+' LCC '+split)
        crop_diagnostics['lcc_'+split]=geometry_summary(g);all_geometry.append(g)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)
    crop_diagnostics['lcc_combined']=geometry_summary(np.concatenate(all_geometry))


def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True)
    print('['+name+'] Starting matched crop',spec['crop_factor'],flush=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping',flush=True)
        return
    model=init_model()
    initial_sha=state_sha(model.state_dict())
    write_json(folder/'initialization_audit.json',{'initial_state_sha256':initial_sha,'train_seed':TRAIN_SEED,'pretrained_checkpoint':str(pretrained)})
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    SWS_SELECTOR_RNG=torch.Generator(device=device).manual_seed(SWS_SELECTOR_SEED) if spec['method']=='SWS' else None
    best_checkpoint=None;crop_diagnostics={'training_epochs':{}}
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state['initial_state_sha256']!=initial_sha:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint'];crop_diagnostics=state['crop_diagnostics']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume committed best differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng,SWS_SELECTOR_RNG)
        print(name,'resume at epoch',start,flush=True)
    val_dl=loader(CelebaData(val,crop_factor=spec['crop_factor']))
    masks=radial_masks(spec['centers'],spec['sigma']) if spec['method']!='CLEAN' else None
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;train_geometry=[];method_samples=[];
            train_ds=CelebaData(train,training=True,epoch=epoch,crop_factor=spec['crop_factor'],scj=spec['method']=='CLEAN')
            train_dl=loader(train_ds,shuffle=True,epoch=epoch);started=time.monotonic()
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, phase={"warmup" if warmup else "active"}, lr={optimizer.param_groups[0]["lr"]:g}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_,g) in enumerate(train_dl,1):
                train_geometry.append(g.numpy())
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,sample_diagnostics=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active,selector_rng=SWS_SELECTOR_RNG)
                if sample_diagnostics:method_samples.append(sample_diagnostics)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-started:.1f}s',flush=True)
            epoch_geometry=np.concatenate(train_geometry)
            method_diagnostics=epoch_method_diagnostics(spec,epoch_geometry,method_samples)
            crop_diagnostics['training_epochs'][str(epoch)]={**geometry_summary(epoch_geometry),**method_diagnostics}
            keys,y,z,g=infer(model,val_dl,label=f'{name} epoch {epoch} Val3K')
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row.update(method_diagnostics)
            diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng,SWS_SELECTOR_RNG),'initial_state_sha256':initial_sha,
                'best_checkpoint':best_checkpoint,'crop_diagnostics':crop_diagnostics,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            write_json(folder/'crop_diagnostics.json',crop_diagnostics)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z,g=infer(model,val_dl,label=name+' frozen Val3K')
    crop_diagnostics['val3k']=geometry_summary(g)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val3k_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val3k_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val3k_metrics':vm,
      'train_crop_factor':spec.get('scj_scale_range',spec['crop_factor']),'eval_crop_factor':EVAL_CROP_FACTOR,'threshold_source':'frozen Val3K at matched crop'}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked,spec['crop_factor'],crop_diagnostics)
    write_json(folder/'crop_diagnostics.json',crop_diagnostics)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'crop_factor':spec['crop_factor'],'initial_state_sha256':initial_sha})
    del model
    torch.cuda.empty_cache()


def scj_crop_scale(epoch,index):
    # Stateless indexed stream: worker scheduling/resume cannot change its draws.
    SCJ_CROP_SCALE_RNG=random.Random(SCJ_SCALE_SEED+epoch*100000+index)
    return SCJ_CROP_SCALE_RNG.uniform(*SCJ_SCALE_RANGE)

def training_crop_scale(training,epoch,index,scj=False):
    return scj_crop_scale(epoch,index) if training and scj else EVAL_CROP_FACTOR

def mc_penalty(clean_margin,spectral_margin):
    gap=F.relu(clean_margin.detach()-spectral_margin)
    return gap.mean(),gap

def sws_probabilities(margins):
    return torch.softmax(-margins.float()/SWS_TAU,dim=1).detach()

def sample_sws_bands(alpha,selector_rng):
    return torch.multinomial(alpha,1,generator=selector_rng).squeeze(1)

def select_soft_worst_view(model,base,binary,masks,gain_rng,selector_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];sampled_margins=[];probabilities=[];hard_bands=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);alpha=sws_probabilities(margin)
                ix=sample_sws_bands(alpha,selector_rng);row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                sampled_margins.append(margin[row,ix].detach());probabilities.append(alpha);hard_bands.append(margin.argmin(1))
    finally:model.train(was_training)
    selected=tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,sampled_margins))
    return selected,torch.cat(probabilities),torch.cat(hard_bands)

def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active,selector_rng=None):
    n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_spectral=None;alpha=None;hard=None;samples={}
    spectral_active=active and spec['method']!='CLEAN'
    if spectral_active:
        if spec['method']=='SWS':
            result,alpha,hard=select_soft_worst_view(model,x,binary,masks,gain_rng,selector_rng)
            selected,bands,gains,_,select_clean,select_spectral=result
        else:
            # Exact notebook13/15 hard P3_SF selection helper, unchanged.
            selected,bands,gains,_,select_clean,select_spectral=select_worst_view(model,x,binary,masks,gain_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;spectral_logits=None;mc=clean_ce*0;gap=None
        if spectral_active:
            # Exactly ONE selected spectral view per sample gets gradient/BN updates.
            spectral_logits=model(selected);spectral_ce=F.cross_entropy(spectral_logits,binary)
            if spec['method']=='MC':mc,gap=mc_penalty(aligned_margin(clean[0],binary),aligned_margin(spectral_logits,binary))
        if not spectral_active:loss=clean_ce+aux_total
        else:loss=spec['clean_ce_weight']*clean_ce+spec['spectral_ce_weight']*spectral_ce+aux_total+spec.get('mc_beta',0.)*mc
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary)
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
          'spectral_ce':float(spectral_ce.detach()),'worst_ce':float(spectral_ce.detach()),'aux_loss':float(aux_total.detach()),
          'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm.mean()),
          'train_clean_aligned_margin':float(cm.mean()),'selection_clean_margin':float(select_clean.mean()) if select_clean is not None else 0.,
          'worst_margin':float(select_spectral.mean()) if select_spectral is not None else 0.,
          'margin_drop':float((select_clean-select_spectral).mean()) if select_spectral is not None else 0.,
          'spectral_flip_rate':flips/n,'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.,
          'gain_mean':float(gains.mean()) if gains is not None else 0.,'mc_loss':float(mc.detach())}
        if spec['method']=='MC' and spectral_active:
            samples={'margin_gap':gap.detach().cpu().numpy(),'clean_margin':cm.cpu().numpy(),
                     'worst_margin':aligned_margin(spectral_logits,binary).cpu().numpy()}
        if alpha is not None:
            samples={'alpha':alpha.cpu().numpy(),'entropy':(-(alpha*alpha.clamp_min(1e-12).log()).sum(1)).cpu().numpy(),
              'sampled_bands':bands.cpu().numpy(),'hard_bands':hard.cpu().numpy()}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts,samples

def distribution_summary(values):
    a=np.asarray(values,dtype=np.float64)
    return {'mean':float(a.mean()),'std':float(a.std()),'min':float(a.min()),'p05':float(np.quantile(a,.05)),
            'median':float(np.median(a)),'p95':float(np.quantile(a,.95)),'max':float(a.max())}

def epoch_method_diagnostics(spec,g,sample_batches):
    result={}
    if spec['method']=='CLEAN':
        for prefix,values in (('requested_scale',g[:,6]),('realized_scale',g[:,3])):
            result.update({prefix+'_'+k:v for k,v in distribution_summary(values).items()})
        result.update(boundary_interaction_fraction=float(g[:,0].mean()),window_translation_fraction=float(g[:,4].mean()),
                      actual_scale_reduction_fraction=float(g[:,5].mean()),requested_scale_center_offset=float(g[:,6].mean()-1.5))
    if sample_batches:
        samples={key:np.concatenate([d[key] for d in sample_batches]) for key in sample_batches[0]}
        if spec['method']=='MC':
            gap=samples['margin_gap'];positive=gap[gap>0]
            result.update(positive_margin_gap_mean=float(positive.mean()) if len(positive) else 0.,
              positive_margin_gap_median=float(np.median(positive)) if len(positive) else 0.,
              positive_margin_gap_max=float(positive.max()) if len(positive) else 0.,margin_gap_active_fraction=float((gap>0).mean()),
              clean_margin_mean=float(samples['clean_margin'].mean()),worst_margin_mean=float(samples['worst_margin'].mean()))
        if spec['method']=='SWS':
            for i,label in enumerate(('low','mid','high')):
                result['alpha_'+label+'_mean']=float(samples['alpha'][:,i].mean())
                result['sampled_'+label+'_fraction']=float((samples['sampled_bands']==i).mean())
                result['hard_'+label+'_fraction']=float((samples['hard_bands']==i).mean())
            result.update(alpha_entropy_mean=float(samples['entropy'].mean()),alpha_entropy_median=float(np.median(samples['entropy'])),
              sampled_matches_hard_argmin_fraction=float((samples['sampled_bands']==samples['hard_bands']).mean()))
    elif spec['method']=='MC':
        result.update({k:0. for k in ('positive_margin_gap_mean','positive_margin_gap_median','positive_margin_gap_max',
                                     'margin_gap_active_fraction','clean_margin_mean','worst_margin_mean')})
    return result

def snapshot_rng(gain_rng,selector_rng=None):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
      'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),
      'sws_selector':selector_rng.get_state() if selector_rng is not None else None,
      'scj_crop_scale':{'seed':SCJ_SCALE_SEED,'mode':'indexed seed+epoch*100000+sample_index'}}

def restore_rng(state,gain_rng,selector_rng=None):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])
    if state['scj_crop_scale']!={'seed':SCJ_SCALE_SEED,'mode':'indexed seed+epoch*100000+sample_index'}:raise RuntimeError('SCJ indexed RNG policy differs')
    if (state['sws_selector'] is None)!=(selector_rng is None):raise RuntimeError('SWS selector resume stream differs')
    if selector_rng is not None:selector_rng.set_state(state['sws_selector'])

def method_forward_sanity_checks():
    # Tiny CPU model only; verifies control flow/BN, not research performance.
    class TinyPAD(nn.Module):
        def __init__(self):
            super().__init__();self.bn=nn.BatchNorm1d(3);self.w=nn.Parameter(torch.tensor(1.));self.calls=[]
        def forward(self,x,aux=False):
            self.calls.append((self.training,torch.is_grad_enabled(),len(x)))
            v=self.bn(x.mean((2,3)));d=(v[:,0]-v[:,1])*self.w;z=torch.stack((d,-d),1)
            return (z,d[:,None].expand(-1,11),d[:,None].expand(-1,5),d[:,None].expand(-1,40)) if aux else z
    global use_amp
    previous_amp=use_amp;use_amp=False
    try:
        with torch.random.fork_rng(devices=[]):
            x=torch.rand(4,3,80,80);y=torch.tensor([0,1,0,1]);labels=torch.zeros(4,dtype=torch.long);attrs=torch.zeros(4,40)
            masks=radial_masks(SCALED_CENTERS,SCALED_SIGMA,device=torch.device('cpu'))
            for name in ('P3_MC','P3_SWS'):
                model=TinyPAD().train();gain=torch.Generator().manual_seed(GAIN_SEED);selector=torch.Generator().manual_seed(SWS_SELECTOR_SEED)
                initial_gain=gain.get_state().clone();initial_selector=selector.get_state().clone()
                loss,scalars,_,samples=training_terms(model,x,y,labels,labels,attrs,RUNS[name],masks,gain,False,selector)
                assert len(model.calls)==1 and int(model.bn.num_batches_tracked)==1 and scalars['mc_loss']==0. and not samples
                assert torch.equal(gain.get_state(),initial_gain) and torch.equal(selector.get_state(),initial_selector)
                assert abs(loss.item()-scalars['clean_ce']-scalars['aux_loss'])<1e-6
                model=TinyPAD().train()
                loss,_,_,_=training_terms(model,x,y,labels,labels,attrs,RUNS[name],masks,gain,True,selector)
                loss.backward()
                gradient_calls=[c for c in model.calls if c[1]];selection_calls=[c for c in model.calls if not c[1]]
                assert len(gradient_calls)==2 and all(c==(True,True,4) for c in gradient_calls)
                assert selection_calls and all(not c[0] for c in selection_calls) and model.training
                assert int(model.bn.num_batches_tracked)==2 and model.w.grad is not None
            model=TinyPAD().train();gain=torch.Generator().manual_seed(GAIN_SEED)
            loss,_,bands,_=training_terms(model,x,y,labels,labels,attrs,RUNS['C_SCJ'],None,gain,True)
            assert len(model.calls)==1 and not bands
            # Full RNG helper replay, including selector, with CUDA state preserved.
            selector=torch.Generator().manual_seed(SWS_SELECTOR_SEED);state=snapshot_rng(gain,selector)
            expected=sample_sws_bands(torch.full((4,3),1/3),selector)
            restore_rng(state,gain,selector)
            assert torch.equal(expected,sample_sws_bands(torch.full((4,3),1/3),selector))
    finally:use_amp=previous_amp
    return {'warmup_no_spectral_MC_selector':True,'BN_safe_selection':True,'one_gradient_spectral_view_per_sample':True,
            'C_SCJ_clean_only':True,'full_selector_snapshot_restore':True}


## Lightweight source-independent sanity checks

No training or dataset inference.

In [ ]:
def implementation_sanity_checks():
    if (MC_BETA,SWS_TAU,SCJ_SCALE_RANGE)!=(.10,1.,(1.30,1.70)):raise RuntimeError('Predefined method constants changed')
    if set(RUNS)!={'C_SCJ','P3_MC','P3_SWS'}:raise RuntimeError('Unexpected experiment matrix')
    py=random.getstate();np_before=np.random.get_state()
    draws=np.array([scj_crop_scale(2,i) for i in range(10000)])
    assert draws.min()>=1.30 and draws.max()<=1.70 and abs(draws.mean()-1.5)<.01
    assert scj_crop_scale(2,7)==scj_crop_scale(2,7) and scj_crop_scale(2,7)!=scj_crop_scale(3,7) and scj_crop_scale(2,7)!=scj_crop_scale(2,8)
    assert random.getstate()==py and np.array_equal(np.random.get_state()[1],np_before[1])
    rec={'bbox_xyxy':[10,20,60,80]};before_box=bbox_jitter(rec,random.Random(BBOX_SEED+200007))
    scj_crop_scale(2,7)
    assert before_box==bbox_jitter(rec,random.Random(BBOX_SEED+200007))
    for epoch in (0,2,20):assert training_crop_scale(False,epoch,7,True)==1.5
    with torch.random.fork_rng(devices=[]):
        clean=torch.tensor([2.,-1.],requires_grad=True);worst=torch.tensor([0.,-2.],requires_grad=True)
        loss,gap=mc_penalty(clean,worst);gc,gw=torch.autograd.grad(loss,(clean,worst),allow_unused=True)
        assert gc is None and torch.equal(gw,torch.full((2,),-.5)) and torch.equal(gap,torch.tensor([2.,1.]))
        cc=torch.tensor(1.2,requires_grad=True);wc=torch.tensor(2.4,requires_grad=True);aux_loss=torch.tensor(.3,requires_grad=True)
        reference=.75*cc+.25*wc+aux_loss;zero_beta=reference+0.*mc_penalty(clean,worst)[0]
        assert torch.equal(reference,zero_beta)
        assert all(torch.equal(a,b) for a,b in zip(torch.autograd.grad(reference,(cc,wc,aux_loss),retain_graph=True),torch.autograd.grad(zero_beta,(cc,wc,aux_loss))))
        margins=torch.tensor([[0.,1.,2.],[3.,0.,-2.]],requires_grad=True)
        alpha=sws_probabilities(margins)
        assert not alpha.requires_grad and torch.isfinite(alpha).all() and torch.allclose(alpha.sum(1),torch.ones(2))
        assert alpha[0,0]>alpha[0,1]>alpha[0,2]
        selector=torch.Generator().manual_seed(SWS_SELECTOR_SEED);saved=selector.get_state().clone()
        ix=sample_sws_bands(alpha,selector);selector.set_state(saved)
        assert torch.equal(ix,sample_sws_bands(alpha,selector)) and ix.shape==(2,) and ((ix>=0)&(ix<3)).all()
    return {'SCJ':{'requested_scale_distribution_seeded_sanity':distribution_summary(draws),'deterministic_indexed_replay':True,
      'independent_bbox_and_augmentation_rng':True,'eval_crop15':True},
      'MC':{'beta_zero_reference_scalar_backward_equivalent':True,'clean_target_detached':True,'worst_gradient_enabled':True},
      'SWS':{'tau':SWS_TAU,'finite_normalized_detached_alpha':True,'lower_margin_higher_probability':True,'selector_rng_replay':True}}

IMPLEMENTATION_CHECKS=implementation_sanity_checks()
IMPLEMENTATION_CHECKS['forward_and_resume']=method_forward_sanity_checks()
write_json(out/'implementation_sanity_checks.json',IMPLEMENTATION_CHECKS)
print('[Sanity]',json.dumps(IMPLEMENTATION_CHECKS,indent=2),flush=True)


## Fixed historical references, Combined-only gate, report and exports

These constants are reporting-only. Pending results are not measured results.

In [ ]:
# Verified historical references supplied in the request: reporting ONLY.
HISTORICAL_C_COMBINED_AUC=.830642
HISTORICAL_P3SF_COMBINED_AUC=.837525
GATE_MIN_COMBINED_AUC=.844642
HISTORICAL_REFERENCES={
 'C_crop15':{'val3k_acer':.021034,'val3k_auc':.996635,'lcc_evaluation_auc':.775915,'lcc_evaluation_hter':.294045,
  'lcc_combined_auc':.830642,'lcc_combined_eer':.244741,'lcc_combined_hter':.245404,'lcc_combined_tpr_at_fpr_1pct':.196291},
 'P3_SF_crop15':{'val3k_acer':.022508,'val3k_auc':.996631,'lcc_evaluation_auc':.782407,'lcc_evaluation_hter':.298970,
  'lcc_combined_auc':.837525,'lcc_combined_eer':.239614,'lcc_combined_hter':.252629,'lcc_combined_tpr_at_fpr_1pct':.202231},
}
METRIC_COLUMNS=('val3k_acer','val3k_auc','lcc_evaluation_auc','lcc_evaluation_hter','lcc_combined_auc',
 'delta_combined_vs_C','delta_combined_vs_P3SF','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')

def gate_status(combined_auc):
    if not np.isfinite(combined_auc):raise ValueError('Gate requires finite Combined AUC')
    if combined_auc>=GATE_MIN_COMBINED_AUC:return 'STRONG_PASS'
    if combined_auc>HISTORICAL_P3SF_COMBINED_AUC:return 'NEAR_MISS'
    return 'NO_STRONG_PASS'

def next_recommendation(rows):
    passing=[r for r in rows if r.get('gate_status')=='STRONG_PASS']
    if not passing:
        near=[r['run_id'] for r in rows if r.get('gate_status')=='NEAR_MISS']
        return 'No STRONG_PASS. NEAR_MISS: '+(', '.join(near) if near else 'none')+'. Decide the next direction manually; no parameter search or extra runs.'
    ordered=sorted(passing,key=lambda r:-r['lcc_combined_auc']);best=ordered[0]
    tied=[r['run_id'] for r in ordered if round(r['lcc_combined_auc'],6)==round(best['lcc_combined_auc'],6)]
    if len(tied)>1:return 'Combined AUC tie at six-decimal reporting precision: '+', '.join(tied)+'. Review manually before selecting one next candidate; no superiority claim or automatic launch.'
    p3=[r for r in ordered if r['run_id'] in ('P3_MC','P3_SWS')]
    if any(r['run_id']=='C_SCJ' for r in passing) and p3:
        return 'Strongest individual passing method: '+best['run_id']+'. At most one possible future combination: SCJ + '+p3[0]['run_id']+' (best passing P3 method). Recommendation only; not implemented or launched.'
    if best['run_id']=='C_SCJ':return 'C_SCJ is the strongest STRONG_PASS. One possible next combination: P3_SF_SCJ, combining scale/context and frequency robustness. Not implemented or launched.'
    return best['run_id']+' is the candidate for later 100K validation. No additional beta/tau search; no automatic launch or statistical superiority claim.'

def export_report():
    rows=[]
    for name in RUNS:
        folder=out/'runs'/name;row={'run_id':name,'status':'pending','gate_status':'PENDING'}
        if (folder/'completion.json').is_file():
            row.update(status='complete',selected_epoch=read_json(folder/'completion.json')['best_epoch'])
            for split in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
                for key,value in read_json(folder/(split+'_metrics.json')).items():row[split+'_'+key]=value
            row['delta_combined_vs_C']=row['lcc_combined_auc']-HISTORICAL_C_COMBINED_AUC
            row['delta_combined_vs_P3SF']=row['lcc_combined_auc']-HISTORICAL_P3SF_COMBINED_AUC
            row['delta_combined_vs_C_pp']=100*row['delta_combined_vs_C']
            row['delta_combined_vs_P3SF_pp']=100*row['delta_combined_vs_P3SF']
            row['gate_status']=gate_status(row['lcc_combined_auc'])
        rows.append(row)
    pd.DataFrame(rows).to_csv(out/'method_discovery_comparison.csv',index=False)
    recommendation=next_recommendation(rows)
    write_json(out/'method_discovery_gate.json',{'metric':'lcc_combined_auc','strong_pass_min':GATE_MIN_COMBINED_AUC,
      'near_miss_strict_min':HISTORICAL_P3SF_COMBINED_AUC,'historical_references':HISTORICAL_REFERENCES,
      'rows':rows,'recommendation':recommendation,'engineering_screen_only':True})
    report=['# Method discovery at crop1.5','',
      'Exactly three hypotheses: C_SCJ context/occupancy sensitivity; P3_MC spectral margin degradation; P3_SWS hard-selection brittleness.',
      'Verified historical references supplied in the request are read-only reporting references; no historical controls are retrained.',
      'Gate uses ONLY LCC Combined AUC: >=0.844642 STRONG_PASS; >0.837525 and <0.844642 NEAR_MISS; otherwise NO_STRONG_PASS.',
      'No secondary metric or diagnostic can override the gate. All training/checkpoint/early-stop/threshold decisions remain source-only.',
      'Best is reloaded, Val3K minACER threshold frozen, physical LCC train/dev/eval inferred at crop1.5 and Combined metrics recomputed from pooled valid scores with coverage accounting.',
      'CelebA full Test evaluation skipped in this micro screening round.','',
      '| Run | '+' | '.join(METRIC_COLUMNS)+' | Gate Status |','|---|'+'---:|'*len(METRIC_COLUMNS)+'---|']
    for row in rows+[dict(run_id=name,gate_status='READ_ONLY_REFERENCE',**values) for name,values in HISTORICAL_REFERENCES.items()]:
        report.append('| '+row['run_id']+' | '+' | '.join(f'{row[k]:.6f}' if k in row else 'pending / not applicable' for k in METRIC_COLUMNS)+' | '+row['gate_status']+' |')
    report+=['','## Combined AUC deltas','']
    for row in rows:
        if row['status']=='complete':report.append('- '+row['run_id']+': versus C '+f'{row["delta_combined_vs_C"]:+.6f} absolute = {row["delta_combined_vs_C_pp"]:+.4f} percentage points; versus P3_SF {row["delta_combined_vs_P3SF"]:+.6f} absolute = {row["delta_combined_vs_P3SF_pp"]:+.4f} percentage points.')
    report+=['','## Method-specific diagnostics','',
      'Epoch diagnostics are empirical descriptions, not causal proof. Positive MC gap mean/median/max are over positive gaps only; mc_loss is the mean over all samples.',
      'SCJ requested-scale center offset is reported empirically; a realized-scale shift from boundary constraints is reported separately.',
      'P3_SWS worst_ce/worst_margin fields retain legacy names and refer to the sampled spectral view, which need not be the hard worst band.','']
    for row in rows:
        if row['status']!='complete':continue
        name=row['run_id'];df=pd.read_csv(out/'runs'/name/'spectral_diagnostics.csv')
        standard=['clean_ce','spectral_ce','train_total_loss','margin_drop','spectral_flip_rate','gain_mean']
        if name=='C_SCJ':keys=[k for k in df if k.startswith(('requested_scale_','realized_scale_'))]+['boundary_interaction_fraction','window_translation_fraction','actual_scale_reduction_fraction']
        elif name=='P3_MC':keys=['mc_loss','positive_margin_gap_mean','positive_margin_gap_median','positive_margin_gap_max','margin_gap_active_fraction','clean_margin_mean','worst_margin_mean']+standard+['low_fraction','mid_fraction','high_fraction']
        else:keys=[k for k in df if k.startswith(('alpha_','sampled_','hard_'))]+standard
        keys=list(dict.fromkeys(k for k in keys if k in df));report+=['### '+name,'','| Epoch | Phase | '+' | '.join(keys)+' |','|---:|---|'+'---:|'*len(keys)]
        for _,d in df.iterrows():report.append('| '+str(int(d.epoch))+' | '+d.phase+' | '+' | '.join(f'{d[k]:.6f}' if pd.notna(d[k]) else 'disabled in warmup' for k in keys)+' |')
        if name=='C_SCJ':
            offset=float(df['requested_scale_mean'].mean()-1.5)
            report.append('\nObserved epoch-average requested-scale offset from1.5: '+f'{offset:+.6f}'+'. Inspect per-epoch quantiles; no distribution adjustment is made.')
        report.append('\nCombined AUC comparison is descriptive: '+row['gate_status']+'. Diagnostics alone do not establish why generalization changed.')
    report+=['','## Recommendation','',recommendation,'','## Limitations','',
      '- Train10K / Val3K micro-scale screening; one training seed.',
      '- LCC has repeatedly been used as an external development benchmark.',
      '- No statistical significance claim; the Combined-AUC gate is a predefined engineering screening criterion.',
      '- 100K validation and full-scale validation have not yet been performed.',
      '- Multi-seed robustness remains future validation because of compute limits.',
      '- New crop/selector streams are independent; SCJ is an indexed deterministic stream and SWS selector state is saved/restored in each run.',
      '- Full GPU execution, actual resource paths, image availability and T4 batch256 memory must be verified on Kaggle.']
    (out/'method_discovery_report.md').write_text('\n'.join(report)+'\n')
    return rows

def zip_outputs(name=None):
    if name is not None:
        folder=out/'runs'/name;run_zip=out.parent/(name+'.zip')
        with zipfile.ZipFile(run_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
            for file in sorted(folder.rglob('*')):
                if file.is_file():archive.write(file,(Path('runs')/name/file.relative_to(folder)).as_posix())
            for filename in ('METHOD_DISCOVERY_CROP15_PROTOCOL_v1.md','implementation_sanity_checks.json'):
                archive.write(out/filename,filename)
        display(FileLink(str(run_zip)))
        return  # Save the standalone ZIP first; overall export follows.
    root_zip=out.parent/'16_micro_method_discovery_crop15.zip'
    print('[Export] Writing:',root_zip,flush=True)
    with zipfile.ZipFile(root_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(root_zip)))

export_report();zip_outputs()


## Train selected runs on Kaggle

This cell starts expensive training; best/threshold decisions use Val3K only.

In [ ]:
selected=select_runs(RUN_FILTER)
print('[Runner] Selected runs:',selected,flush=True)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch256 ran out of GPU memory; stop instead of changing the recipe') from exc
    # Archive the completed run first; subsequent reporting/failure cannot remove it.
    zip_outputs(name)
    export_report();zip_outputs()
print('Finished selected method-discovery runs:',selected,flush=True)
for name in selected:display(FileLink(str(out.parent/(name+'.zip'))))
display(FileLink(str(out.parent/'16_micro_method_discovery_crop15.zip')))
